In [17]:
# ================================
# F1 Pit Stop Prediction - Kaggle
# Complete Training Pipeline
# ================================

# Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

# ================================
# Load Dataset
# ================================

train_df = pd.read_csv('D:\\DS Part(2)\\F1_pitstop\\playground-series-s6e5\\train.csv')
test_df = pd.read_csv('D:\\DS Part(2)\\F1_pitstop\\playground-series-s6e5\\test.csv')

# ================================
# Save Test IDs
# ================================

test_ids = test_df['id']

# ================================
# Drop ID Column
# ================================

train_df.drop('id', axis=1, inplace=True)
test_df.drop('id', axis=1, inplace=True)

# ================================
# Encode Categorical Features
# ================================

cat_cols = train_df.select_dtypes(include='object').columns

label_encoders = {}

for col in cat_cols:
    
    le = LabelEncoder()
    
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    label_encoders[col] = le

# ================================
# Feature Engineering
# ================================

train_df['WearRate'] = (
    train_df['Cumulative_Degradation'] /
    (train_df['TyreLife'] + 1)
)

test_df['WearRate'] = (
    test_df['Cumulative_Degradation'] /
    (test_df['TyreLife'] + 1)
)

train_df['PaceDrop'] = (
    train_df['LapTime_Delta'] /
    (train_df['TyreLife'] + 1)
)

test_df['PaceDrop'] = (
    test_df['LapTime_Delta'] /
    (test_df['TyreLife'] + 1)
)

# ================================
# Remove Highly Correlated Feature
# ================================

train_df.drop('LapNumber', axis=1, inplace=True)
test_df.drop('LapNumber', axis=1, inplace=True)

# ================================
# Separate Features & Target
# ================================



In [20]:
train_df.columns

Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'Stint', 'TyreLife',
       'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
       'RaceProgress', 'Position_Change', 'PitNextLap', 'WearRate',
       'PaceDrop'],
      dtype='object')

In [21]:
X = train_df.drop('PitNextLap', axis=1)
y = train_df['PitNextLap']

In [22]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ================================
# Build Model
# ================================

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

# ================================
# Train Model
# ================================

model.fit(X_train, y_train)

# ================================
# Validation Prediction
# ================================

valid_pred = model.predict(X_valid)

# ================================
# Evaluation
# ================================

accuracy = accuracy_score(y_valid, valid_pred)

print(f"Validation Accuracy: {accuracy:.4f}")

print("\nClassification Report:\n")

print(classification_report(y_valid, valid_pred))

# ================================
# Train On Full Dataset
# ================================

model.fit(X, y)

# ================================
# Predict Test Dataset
# ================================

test_pred = model.predict(test_df)

# ================================
# Create Submission File
# ================================

submission = pd.DataFrame({
    'id': test_ids,
    'PitNextLap': test_pred
})

# ================================
# Save Submission
# ================================

submission.to_csv(
    'submission.csv',
    index=False
)

print("\nsubmission.csv created successfully!")

Validation Accuracy: 0.8956

Classification Report:

              precision    recall  f1-score   support

         0.0       0.93      0.94      0.94     70352
         1.0       0.75      0.72      0.73     17476

    accuracy                           0.90     87828
   macro avg       0.84      0.83      0.83     87828
weighted avg       0.89      0.90      0.89     87828


submission.csv created successfully!
